# SARSA from Scratch: CliffWalking — with Side-by-Side Q-Learning Comparison

This notebook implements both **SARSA** and **Q-learning** completely from scratch (no RL libraries).

**What you will build:**
1. CliffWalking environment walkthrough and grid visualizer
2. SARSA agent (on-policy TD) from scratch
3. Q-learning agent (off-policy TD) from scratch — sharing the same code skeleton
4. Side-by-side training with identical hyperparameters
5. **The critical experiment**: why SARSA takes the safe path and Q-learning takes the risky path
6. On-policy vs off-policy demonstrated visually via learned Q-values and policies
7. Expected SARSA implementation and comparison
8. Comprehensive quiz in code form — predict before running!


## Cell 1 — Imports

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

print('All imports successful.')
print(f'NumPy: {np.__version__}')


## Cell 2 — CliffWalking Environment

The grid is 4 rows × 12 columns = 48 states (numbered left-to-right, top-to-bottom).

```
 Col:  0   1   2   3   4   5   6   7   8   9  10  11
 Row 0: 0   1   2   3   4   5   6   7   8   9  10  11
 Row 1:12  13  14  15  16  17  18  19  20  21  22  23
 Row 2:24  25  26  27  28  29  30  31  32  33  34  35
 Row 3:S  [C   C   C   C   C   C   C   C   C   C]  G
       36  37  38  39  40  41  42  43  44  45  46  47
```

- **S** (36) = Start | **G** (47) = Goal | **C** (37-46) = Cliff
- Stepping on cliff → reward −100 + reset to Start (episode continues!)
- Every other step → reward −1
- Reaching Goal → episode ends


In [ ]:
env = gym.make('CliffWalking-v1')
obs, _ = env.reset(seed=SEED)

print('=== CliffWalking-v1 ===')
print(f'State space : Discrete({env.observation_space.n})  (48 cells)')
print(f'Action space: Discrete({env.action_space.n})  (0=Up, 1=Right, 2=Down, 3=Left)')
print(f'Grid shape  : {env.unwrapped.shape}  (4 rows × 12 cols)')
print(f'Start state : 36  (row 3, col 0)')
print(f'Goal  state : 47  (row 3, col 11)')
print(f'Cliff states: 37-46  (row 3, cols 1-10)')
print()

# --- Constants ---
NROW, NCOL = 4, 12
N_STATES   = 48
N_ACTIONS  = 4
START      = 36
GOAL       = 47
CLIFF      = set(range(37, 47))
ACTION_NAMES = ['Up', 'Right', 'Down', 'Left']
ACTION_DELTAS = {0: (-1,0), 1: (0,1), 2: (1,0), 3: (0,-1)}

def state_to_rc(s):
    return s // NCOL, s % NCOL

def rc_to_state(r, c):
    return r * NCOL + c

# --- Verify key mechanics ---
print('Mechanics verification:')
env.reset()

# Safe step (up)
for _ in range(3): env.step(2)   # move to row 3
s, r, t, _, _ = env.step(0)
print(f'  Up from row 3, col 0: → state={s} (row {s//12}, col {s%12}), reward={r}')

env.reset()
for _ in range(3): env.step(2)   # row 3
s, r, t, _, _ = env.step(1)     # right → cliff!
print(f'  Right into cliff   : → state={s} (reset to Start), reward={r}')

env.close()


## Cell 3 — Grid Visualizer

In [ ]:
def plot_grid(title='CliffWalking Grid', path=None, q_values=None, policy=None,
              highlight_states=None, ax=None):
    """
    Visualize the CliffWalking grid.
    
    Parameters
    ----------
    path             : list of states (draw trajectory)
    q_values         : (48, 4) array — draw max Q as heatmap
    policy           : (48,) int array — draw arrows
    highlight_states : dict {state: color} for special cells
    """
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(14, 5))
    
    # Base grid colors
    grid_img = np.zeros((NROW, NCOL, 3))
    for r in range(NROW):
        for c in range(NCOL):
            s = rc_to_state(r, c)
            if s in CLIFF:
                grid_img[r, c] = [0.2, 0.2, 0.2]     # dark — cliff
            elif s == GOAL:
                grid_img[r, c] = [0.1, 0.7, 0.2]     # green — goal
            elif s == START:
                grid_img[r, c] = [0.2, 0.4, 0.8]     # blue — start
            else:
                if q_values is not None:
                    v = np.max(q_values[s])
                    norm_v = (v - q_values.max()) / (q_values.max() - q_values.min() + 1e-9)
                    intensity = 0.3 + 0.6 * (1 + norm_v)
                    intensity = np.clip(intensity, 0.3, 0.95)
                    grid_img[r, c] = [intensity * 0.9, intensity * 0.85, intensity]
                else:
                    grid_img[r, c] = [0.92, 0.92, 0.95]  # light gray
    
    ax.imshow(grid_img, aspect='auto', interpolation='nearest')
    
    # Grid lines
    for x in range(NCOL + 1):
        ax.axvline(x - 0.5, color='white', linewidth=0.8)
    for y in range(NROW + 1):
        ax.axhline(y - 0.5, color='white', linewidth=0.8)
    
    # Cell labels
    arrow_dirs = {0: (0, -0.35), 1: (0.35, 0), 2: (0, 0.35), 3: (-0.35, 0)}
    for r in range(NROW):
        for c in range(NCOL):
            s = rc_to_state(r, c)
            if s in CLIFF:
                ax.text(c, r, '▼', ha='center', va='center', fontsize=10,
                        color='red', fontweight='bold')
            elif s == GOAL:
                ax.text(c, r, 'G', ha='center', va='center', fontsize=11,
                        color='white', fontweight='bold')
            elif s == START:
                ax.text(c, r, 'S', ha='center', va='center', fontsize=11,
                        color='white', fontweight='bold')
            else:
                ax.text(c, r, str(s), ha='center', va='center',
                        fontsize=6, color='#555555')
            
            # Policy arrows
            if policy is not None and s not in CLIFF and s != GOAL:
                a = policy[s]
                dx, dy = arrow_dirs[a]
                ax.annotate('', xy=(c+dx, r+dy), xytext=(c, r),
                            arrowprops=dict(arrowstyle='->', color='#cc2200',
                                           lw=1.8), zorder=4)
    
    # Trajectory
    if path:
        path_r = [state_to_rc(s)[0] for s in path]
        path_c = [state_to_rc(s)[1] for s in path]
        ax.plot(path_c, path_r, 'yo-', linewidth=2, markersize=4,
                alpha=0.8, zorder=3)
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xticks(range(NCOL))
    ax.set_xticklabels([f'c{i}' for i in range(NCOL)], fontsize=8)
    ax.set_yticks(range(NROW))
    ax.set_yticklabels([f'r{i}' for i in range(NROW)], fontsize=8)
    
    if standalone:
        plt.tight_layout()
        plt.show()

plot_grid('CliffWalking Grid (▼=Cliff, S=Start, G=Goal)')


## Cell 4 — SARSA Agent: On-Policy TD Control

The SARSA loop structure is the key implementation detail:

```
state ← env.reset()
action ← ε-greedy(Q, state)          ← select FIRST action up front

while not done:
    next_state, reward ← env.step(action)
    next_action ← ε-greedy(Q, next_state)  ← select NEXT action BEFORE update
    
    # (S, A, R, S', A') — all 5 used here
    Q[state, action] += α · [r + γ·Q[next_state, next_action] − Q[state, action]]
    
    state  ← next_state
    action ← next_action                   ← carry next_action forward
```

`next_action` is selected BEFORE the update and carried forward — it IS the action
the agent will take next. This is what makes it on-policy.


In [ ]:
class SARSAAgent:
    """
    SARSA (on-policy TD control).
    
    On-policy: the Q-values learned represent the value of the ε-greedy
    behavior policy — including its random exploration steps.
    """
    
    def __init__(self, n_states, n_actions, alpha=0.5, gamma=0.99,
                 epsilon=0.1, epsilon_decay=1.0, epsilon_min=0.01):
        self.n_states    = n_states
        self.n_actions   = n_actions
        self.alpha       = alpha
        self.gamma       = gamma
        self.epsilon     = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min   = epsilon_min
        
        # Q-table: shape (n_states, n_actions), initialized to 0
        self.Q = np.zeros((n_states, n_actions))
        
        # Diagnostics
        self.td_errors   = []
    
    def epsilon_greedy(self, state):
        """ε-greedy action selection."""
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        return np.argmax(self.Q[state])
    
    def update(self, s, a, r, s_next, a_next, done):
        """
        SARSA update: uses ACTUAL next action a_next (on-policy).
        
        Q(s,a) ← Q(s,a) + α·[r + γ·Q(s',a') − Q(s,a)]
        
        Note: if done (terminal), future value = 0
        """
        current_q  = self.Q[s, a]
        future_q   = 0.0 if done else self.Q[s_next, a_next]
        td_error   = r + self.gamma * future_q - current_q
        self.Q[s, a] += self.alpha * td_error
        self.td_errors.append(abs(td_error))
        return td_error
    
    def run_episode(self, env):
        """
        Run one complete episode using SARSA.
        Returns total reward and trajectory (list of states).
        """
        state, _ = env.reset()
        # KEY: select FIRST action before entering loop
        action   = self.epsilon_greedy(state)
        
        total_reward = 0
        trajectory   = [state]
        cliff_falls  = 0
        
        for _ in range(10_000):
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            # KEY: select NEXT action BEFORE update
            next_action = self.epsilon_greedy(next_state)
            
            # SARSA update: (S, A, R, S', A')
            self.update(state, action, reward, next_state, next_action, done)
            
            if reward == -100:
                cliff_falls += 1
            
            trajectory.append(next_state)
            total_reward += reward
            state  = next_state
            action = next_action   # carry forward
            
            if done:
                break
        
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
        return total_reward, trajectory, cliff_falls
    
    def get_policy(self):
        """Greedy policy: argmax Q for each state."""
        return np.argmax(self.Q, axis=1)


# Quick test
env_test = gym.make('CliffWalking-v1')
sarsa_test = SARSAAgent(N_STATES, N_ACTIONS, alpha=0.5, gamma=0.99, epsilon=0.1)
r, traj, falls = sarsa_test.run_episode(env_test)
print(f'SARSA test episode: reward={r:.0f}, steps={len(traj)}, cliff falls={falls}')
print(f'Q-table shape: {sarsa_test.Q.shape}')
print(f'Sample Q-values at START (36): {sarsa_test.Q[START].round(4)}')
env_test.close()


## Cell 5 — Q-Learning Agent: Off-Policy TD Control

Q-learning differs from SARSA in **one line only** — the update uses `max Q(s', ·)` instead of `Q(s', a')`.

The structural implication: `action` is re-selected at the **top of the loop** each step, so no "carry-forward" of `next_action` is needed.

```
while not done:
    action ← ε-greedy(Q, state)
    next_state, reward ← env.step(action)
    
    # Off-policy: imagines the best action at s' (not necessarily what we'll do)
    Q[state, action] += α · [r + γ·max Q[next_state, :] − Q[state, action]]
    
    state ← next_state
```


In [ ]:
class QLearningAgent:
    """
    Q-learning (off-policy TD control).
    
    Off-policy: Q-values represent the value of the GREEDY (optimal) policy,
    even though the agent follows ε-greedy during training.
    """
    
    def __init__(self, n_states, n_actions, alpha=0.5, gamma=0.99,
                 epsilon=0.1, epsilon_decay=1.0, epsilon_min=0.01):
        self.n_states    = n_states
        self.n_actions   = n_actions
        self.alpha       = alpha
        self.gamma       = gamma
        self.epsilon     = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min   = epsilon_min
        self.Q = np.zeros((n_states, n_actions))
        self.td_errors = []
    
    def epsilon_greedy(self, state):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        return np.argmax(self.Q[state])
    
    def update(self, s, a, r, s_next, done):
        """
        Q-learning update: uses MAX Q at next state (off-policy).
        
        Q(s,a) ← Q(s,a) + α·[r + γ·max_a' Q(s',a') − Q(s,a)]
        
        No a_next needed — imagines the best action at s'.
        """
        current_q  = self.Q[s, a]
        future_q   = 0.0 if done else np.max(self.Q[s_next])   # ← THE difference
        td_error   = r + self.gamma * future_q - current_q
        self.Q[s, a] += self.alpha * td_error
        self.td_errors.append(abs(td_error))
        return td_error
    
    def run_episode(self, env):
        state, _ = env.reset()
        total_reward = 0
        trajectory   = [state]
        cliff_falls  = 0
        
        for _ in range(10_000):
            action = self.epsilon_greedy(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            # Q-learning update: (S, A, R, S') — no a_next
            self.update(state, action, reward, next_state, done)
            
            if reward == -100:
                cliff_falls += 1
            
            trajectory.append(next_state)
            total_reward += reward
            state = next_state
            
            if done:
                break
        
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
        return total_reward, trajectory, cliff_falls
    
    def get_policy(self):
        return np.argmax(self.Q, axis=1)


print('=== CODE COMPARISON ===')
print()
print('SARSA update:')
print('  future_q = Q[s_next, a_next]       ← uses ACTUAL sampled next action')
print()
print('Q-learning update:')
print('  future_q = np.max(Q[s_next])       ← uses MAX over all next actions')
print()
print('Everything else — epsilon_greedy, Q-table, alpha, gamma — is IDENTICAL.')


## Cell 6 — Step-by-Step Trace: One Update from Each Algorithm

Before training, trace through one update for each algorithm on the same transition to see the difference concretely.


In [ ]:
np.random.seed(SEED)
print('=== ONE UPDATE TRACE: SARSA vs Q-LEARNING ===')
print()

# Synthetic scenario: agent is at state 24 (row 2, col 0), one step above Start
# It moves Right to state 25. Next, policy says move Down (toward cliff!).
# But actual next action chosen: Right again (safer).

# Manually set some Q-values to make the example interesting
Q_demo = np.zeros((N_STATES, N_ACTIONS))
# At state 25, pretend Q-values are:
# Up=−5, Right=−2, Down=−50 (high cliff risk), Left=−8
Q_demo[25] = [-5.0, -2.0, -50.0, -8.0]

alpha = 0.5; gamma = 0.99

# Transition: s=24, a=Right(1), r=-1, s'=25
s, a, r, s_next = 24, 1, -1.0, 25
a_next_sarsa = 1    # ε-greedy chose Right (sensible)
a_next_greedy = 1   # greedy also picks Right (max Q at 25)

print(f'Transition: s={s} (r2,c0)  →  a=Right  →  r={r}  →  s'={s_next} (r2,c1)')
print(f'Q[s, a]  = Q[{s}, Right] = {Q_demo[s, a]:.2f}  (initialized to 0)')
print()
print(f'Q[s', :] = {Q_demo[s_next].round(2)}  (Up, Right, Down, Left)')
print(f'  max Q[s'] = {np.max(Q_demo[s_next]):.2f}  (Right, index 1)')
print(f'  Q[s', a_next_sarsa=Right] = {Q_demo[s_next, a_next_sarsa]:.2f}')
print()

# SARSA update
td_sarsa    = r + gamma * Q_demo[s_next, a_next_sarsa] - Q_demo[s, a]
new_q_sarsa = Q_demo[s, a] + alpha * td_sarsa

# Q-learning update
td_qlearn    = r + gamma * np.max(Q_demo[s_next]) - Q_demo[s, a]
new_q_qlearn = Q_demo[s, a] + alpha * td_qlearn

print('─' * 60)
print('SARSA update:')
print(f'  TD error = r + γ·Q[s',a_next] − Q[s,a]')
print(f'           = {r} + {gamma}·{Q_demo[s_next,a_next_sarsa]:.2f} − {Q_demo[s,a]:.2f}')
print(f'           = {td_sarsa:.4f}')
print(f'  New Q[{s},Right] = {Q_demo[s,a]:.2f} + {alpha}·{td_sarsa:.4f} = {new_q_sarsa:.4f}')
print()
print('Q-learning update:')
print(f'  TD error = r + γ·max Q[s'] − Q[s,a]')
print(f'           = {r} + {gamma}·{np.max(Q_demo[s_next]):.2f} − {Q_demo[s,a]:.2f}')
print(f'           = {td_qlearn:.4f}')
print(f'  New Q[{s},Right] = {Q_demo[s,a]:.2f} + {alpha}·{td_qlearn:.4f} = {new_q_qlearn:.4f}')
print()
print('─' * 60)
print('Difference in this step:', abs(new_q_sarsa - new_q_qlearn))
print()
print('In this case both chose the same a_next=Right, so result is identical.')
print()
print('NOW — the critical case: a_next_sarsa is forced to be DOWN (random exploration):')
a_next_random = 2   # Down — toward cliff
td_sarsa_risky = r + gamma * Q_demo[s_next, a_next_random] - Q_demo[s, a]
new_q_risky    = Q_demo[s, a] + alpha * td_sarsa_risky
print(f'  SARSA a_next=Down: TD = {r} + {gamma}·{Q_demo[s_next,a_next_random]:.2f} − 0')
print(f'  = {td_sarsa_risky:.4f}  → new Q = {new_q_risky:.4f}')
print()
print('  Q-learning always uses max = −2.00 (Right) regardless of what the policy does.')
print(f'  SARSA here uses −50.0 (Down — explored randomly) → much lower Q update.')
print()
print('This is the mechanism: SARSA "feels" the pain of risky exploration.')
print('Q-learning ignores exploratory mistakes in its update.')


## Cell 7 — Training: SARSA vs Q-Learning Side-by-Side

Both agents trained with identical hyperparameters for 500 episodes.
We track per-episode reward and cliff falls to compare online performance.


In [ ]:
N_EPISODES  = 500
ALPHA       = 0.5
GAMMA       = 0.99
EPSILON     = 0.1   # fixed (no decay) to keep on-policy distinction clear

np.random.seed(SEED)
env_sarsa = gym.make('CliffWalking-v1')
env_qlearn = gym.make('CliffWalking-v1')

sarsa_agent  = SARSAAgent(N_STATES, N_ACTIONS, alpha=ALPHA, gamma=GAMMA, epsilon=EPSILON)
qlearn_agent = QLearningAgent(N_STATES, N_ACTIONS, alpha=ALPHA, gamma=GAMMA, epsilon=EPSILON)

sarsa_rewards  = []; sarsa_cliff_falls  = []
qlearn_rewards = []; qlearn_cliff_falls = []

print(f'Training for {N_EPISODES} episodes (ε={EPSILON} fixed, α={ALPHA}, γ={GAMMA})')
print(f'{"Episode":>8}  {"SARSA_R":>10}  {"QL_R":>10}  {"SARSA_Falls":>12}  {"QL_Falls":>10}')
print('─' * 55)

for ep in range(N_EPISODES):
    sr, _, scf = sarsa_agent.run_episode(env_sarsa)
    qr, _, qcf = qlearn_agent.run_episode(env_qlearn)
    
    sarsa_rewards.append(sr);   sarsa_cliff_falls.append(scf)
    qlearn_rewards.append(qr);  qlearn_cliff_falls.append(qcf)
    
    if (ep + 1) % 100 == 0:
        sm = np.mean(sarsa_rewards[-50:]);  qm = np.mean(qlearn_rewards[-50:])
        sc = np.sum(sarsa_cliff_falls[-50:])
        qc = np.sum(qlearn_cliff_falls[-50:])
        print(f'{ep+1:>8d}  {sm:>10.1f}  {qm:>10.1f}  {sc:>12d}  {qc:>10d}')

env_sarsa.close()
env_qlearn.close()
print()
print(f'Final 100-ep mean reward — SARSA: {np.mean(sarsa_rewards[-100:]):.1f}  |  Q-learning: {np.mean(qlearn_rewards[-100:]):.1f}')
print(f'Total cliff falls (all eps)  — SARSA: {sum(sarsa_cliff_falls)}  |  Q-learning: {sum(qlearn_cliff_falls)}')


## Cell 8 — Training Curves: The Online Performance Gap

This is the central experiment. Four plots:
1. Reward per episode (smoothed)
2. Cliff falls per episode
3. Cumulative cliff falls
4. Reward distribution

**What to look for:** SARSA should have higher (less negative) average reward during training despite converging to a slightly longer path, because it falls off the cliff far less.


In [ ]:
def smooth(data, w=20):
    if len(data) < w: return np.array(data)
    return np.convolve(data, np.ones(w)/w, mode='valid')

fig = plt.figure(figsize=(15, 10))
fig.suptitle('SARSA vs Q-Learning on CliffWalking-v1\n(On-Policy vs Off-Policy Training Behaviour)',
             fontsize=14, fontweight='bold')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.38)
eps = np.arange(N_EPISODES)

# 1. Reward per episode
ax1 = fig.add_subplot(gs[0, :2])
ax1.plot(eps, sarsa_rewards,  alpha=0.15, color='steelblue')
ax1.plot(eps, qlearn_rewards, alpha=0.15, color='coral')
sm_s = smooth(sarsa_rewards,  20)
sm_q = smooth(qlearn_rewards, 20)
ax1.plot(np.arange(len(sm_s)), sm_s, color='steelblue', linewidth=2.5,
         label=f'SARSA (final mean: {np.mean(sarsa_rewards[-100:]):.0f})')
ax1.plot(np.arange(len(sm_q)), sm_q, color='coral',     linewidth=2.5,
         label=f'Q-Learning (final mean: {np.mean(qlearn_rewards[-100:]):.0f})')
ax1.axhline(-13, color='steelblue', linestyle='--', alpha=0.5, label='Safe path (−13)')
ax1.axhline(-12, color='coral',     linestyle='--', alpha=0.5, label='Risky path (−12)')
ax1.set_title('Episode Reward (smoothed 20-ep)')
ax1.set_xlabel('Episode'); ax1.set_ylabel('Total Reward')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)
ax1.set_ylim(-500, 0)

# 2. Cliff falls per episode
ax2 = fig.add_subplot(gs[0, 2])
sm_sc = smooth(sarsa_cliff_falls,  20)
sm_qc = smooth(qlearn_cliff_falls, 20)
ax2.plot(np.arange(len(sm_sc)), sm_sc, color='steelblue', linewidth=2, label='SARSA')
ax2.plot(np.arange(len(sm_qc)), sm_qc, color='coral',     linewidth=2, label='Q-Learning')
ax2.set_title('Cliff Falls per Episode\n(smoothed 20-ep)')
ax2.set_xlabel('Episode'); ax2.set_ylabel('Falls')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

# 3. Cumulative cliff falls
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(eps, np.cumsum(sarsa_cliff_falls),  color='steelblue', linewidth=2, label=f'SARSA (total: {sum(sarsa_cliff_falls)})')
ax3.plot(eps, np.cumsum(qlearn_cliff_falls), color='coral',     linewidth=2, label=f'Q-Learning (total: {sum(qlearn_cliff_falls)})')
ax3.set_title('Cumulative Cliff Falls')
ax3.set_xlabel('Episode'); ax3.set_ylabel('Total Falls')
ax3.legend(fontsize=9); ax3.grid(True, alpha=0.3)

# 4. Reward distribution (last 200 eps)
ax4 = fig.add_subplot(gs[1, 1])
ax4.hist(sarsa_rewards[-200:],  bins=30, alpha=0.7, color='steelblue', label='SARSA')
ax4.hist(qlearn_rewards[-200:], bins=30, alpha=0.7, color='coral',     label='Q-Learning')
ax4.axvline(-13, color='steelblue', linestyle='--', alpha=0.7, label='Safe (−13)')
ax4.axvline(-12, color='coral',     linestyle='--', alpha=0.7, label='Risky (−12)')
ax4.set_title('Reward Distribution (last 200 eps)')
ax4.set_xlabel('Reward'); ax4.set_ylabel('Count')
ax4.legend(fontsize=8); ax4.grid(True, alpha=0.3)

# 5. TD error convergence
ax5 = fig.add_subplot(gs[1, 2])
s_td = smooth([abs(e) for e in sarsa_agent.td_errors[-5000:]], 50)
q_td = smooth([abs(e) for e in qlearn_agent.td_errors[-5000:]], 50)
ax5.plot(s_td, color='steelblue', linewidth=1.5, alpha=0.9, label='SARSA |TD error|')
ax5.plot(q_td, color='coral',     linewidth=1.5, alpha=0.9, label='Q-Learning |TD error|')
ax5.set_title('|TD Error| (last 5000 updates)')
ax5.set_xlabel('Update step'); ax5.legend(fontsize=8); ax5.grid(True, alpha=0.3)

plt.show()

print('Key takeaway:')
print(f'  SARSA total cliff falls   : {sum(sarsa_cliff_falls):4d}  (safer training)')
print(f'  Q-learning total cliff falls: {sum(qlearn_cliff_falls):4d}  (more dangerous training)')
print(f'  SARSA avg reward (last 100): {np.mean(sarsa_rewards[-100:]):.1f}  (safe path)')
print(f'  Q-learn avg reward (last 100): {np.mean(qlearn_rewards[-100:]):.1f}  (risky path)')


## Cell 9 — The Learned Policies: Safe vs Risky

The most vivid demonstration: plot the greedy policy arrow map for each algorithm.
SARSA's optimal policy should route through row 2 (safe). Q-learning's should hug row 3.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

policy_sarsa  = sarsa_agent.get_policy()
policy_qlearn = qlearn_agent.get_policy()

plot_grid('SARSA Policy (On-Policy) — Should take SAFE path',
          q_values=sarsa_agent.Q, policy=policy_sarsa, ax=axes[0])
plot_grid('Q-Learning Policy (Off-Policy) — Should take RISKY path',
          q_values=qlearn_agent.Q, policy=policy_qlearn, ax=axes[1])

plt.tight_layout()
plt.show()

# Print the policy path each would take from Start
print('=== POLICY PATH TRACE: What path does each greedy policy take? ===')
print()

def trace_greedy_path(policy, max_steps=50):
    state = START
    path  = [state]
    for _ in range(max_steps):
        action = policy[state]
        # Manual step (no env needed)
        r, c = state_to_rc(state)
        dr, dc = {0:(-1,0), 1:(0,1), 2:(1,0), 3:(0,-1)}[action]
        nr, nc = max(0,min(NROW-1,r+dr)), max(0,min(NCOL-1,c+dc))
        state = rc_to_state(nr, nc)
        if state in CLIFF:
            path.append(state)
            print(f'  ⚠ Fell off cliff at state {state}!')
            break
        path.append(state)
        if state == GOAL:
            break
    return path

sarsa_path  = trace_greedy_path(policy_sarsa)
qlearn_path = trace_greedy_path(policy_qlearn)

def describe_path(path):
    steps = []
    for s in path:
        r, c = state_to_rc(s)
        marker = 'S' if s==START else ('G' if s==GOAL else ('C' if s in CLIFF else f'({r},{c})'))
        steps.append(marker)
    return ' → '.join(steps)

print(f'SARSA  greedy path ({len(sarsa_path)-1} steps): {describe_path(sarsa_path)}')
print()
print(f'Q-learn greedy path ({len(qlearn_path)-1} steps): {describe_path(qlearn_path)}')


## Cell 10 — Q-Value Heatmaps: See the Risk Encoded in the Values

Compare the Q-values each algorithm assigns to cliff-adjacent states.
SARSA's Q-values near the cliff (row 3) should be low (risk-aware).
Q-learning's should be higher (ignores exploration risk).


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
fig.suptitle('Max Q-Values per Cell: SARSA vs Q-Learning\n'
             '(Brighter = higher value = agent considers this state more valuable)',
             fontsize=13, fontweight='bold')

action_names_short = ['↑','→','↓','←']

for row_idx, (agent_obj, agent_name) in enumerate([(sarsa_agent,'SARSA'), (qlearn_agent,'Q-Learning')]):
    # Max Q heatmap
    ax = axes[row_idx, 0]
    max_q = np.max(agent_obj.Q, axis=1).reshape(NROW, NCOL)
    # Mask cliff
    for cs in CLIFF:
        r, c = state_to_rc(cs)
        max_q[r, c] = np.nan
    
    im = ax.imshow(max_q, cmap='RdYlGn', interpolation='nearest', vmin=-50, vmax=0)
    plt.colorbar(im, ax=ax, shrink=0.8)
    for r in range(NROW):
        for c in range(NCOL):
            s = rc_to_state(r, c)
            if s in CLIFF:
                ax.add_patch(plt.Rectangle((c-0.5,r-0.5),1,1,color='black',zorder=2))
                ax.text(c, r, '▼', ha='center', va='center', color='red', fontsize=10, zorder=3)
            elif s == GOAL:
                ax.text(c, r, 'G', ha='center', va='center', color='white', fontsize=10, fontweight='bold')
            elif s == START:
                ax.text(c, r, 'S', ha='center', va='center', color='white', fontsize=10, fontweight='bold')
            else:
                ax.text(c, r, f'{np.max(agent_obj.Q[s]):.1f}', ha='center', va='center', fontsize=7)
    ax.set_title(f'{agent_name}: max Q(s, ·)')
    ax.set_xticks(range(NCOL)); ax.set_xticklabels([f'c{i}' for i in range(NCOL)], fontsize=7)
    ax.set_yticks(range(NROW)); ax.set_yticklabels([f'r{i}' for i in range(NROW)], fontsize=8)
    
    # Row 3 Q-values detail
    ax2 = axes[row_idx, 1]
    row3_states = [rc_to_state(3, c) for c in range(NCOL)]
    row3_q = agent_obj.Q[row3_states]
    
    x = np.arange(NCOL)
    width = 0.2
    colors = ['#3a86ff','#8338ec','#ff006e','#fb5607']
    for i, aname in enumerate(action_names_short):
        ax2.bar(x + i*width - 1.5*width, row3_q[:, i], width, label=f'{aname}', color=colors[i], alpha=0.85)
    ax2.axhline(0, color='black', linewidth=0.8)
    for cliff_c in range(1, 11):
        ax2.axvline(cliff_c - 0.5, color='red', alpha=0.2, linewidth=8)
    ax2.set_title(f'{agent_name}: Q-values at Row 3 (cliff row)\n(red shading = cliff cells)')
    ax2.set_xlabel('Column'); ax2.set_ylabel('Q-value')
    ax2.set_xticks(range(NCOL)); ax2.set_xticklabels([f'c{i}' for i in range(NCOL)], fontsize=8)
    ax2.legend(fontsize=8, loc='upper left'); ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


## Cell 11 — The ε Experiment: How Exploration Level Affects Each Algorithm

**Hypothesis:** The SARSA/Q-learning gap should be larger at high ε (more exploration → more cliff risk for Q-learning) and smaller at low ε (less exploration → both behave similarly).

We train both algorithms at 4 different ε values and compare cliff falls and final rewards.


In [ ]:
epsilons = [0.01, 0.05, 0.1, 0.3]
results = {}

print('Training at different ε values (200 episodes each)...')
print(f'{"ε":>6}  {"SARSA reward":>14}  {"QL reward":>12}  {"SARSA falls":>13}  {"QL falls":>10}')
print('─' * 60)

for eps_val in epsilons:
    np.random.seed(SEED)
    e_sarsa  = gym.make('CliffWalking-v1')
    e_qlearn = gym.make('CliffWalking-v1')
    s_ag = SARSAAgent(N_STATES, N_ACTIONS, alpha=0.5, gamma=0.99, epsilon=eps_val)
    q_ag = QLearningAgent(N_STATES, N_ACTIONS, alpha=0.5, gamma=0.99, epsilon=eps_val)
    s_r, q_r, s_cf, q_cf = [], [], [], []
    for _ in range(200):
        r, _, cf = s_ag.run_episode(e_sarsa);  s_r.append(r);  s_cf.append(cf)
        r, _, cf = q_ag.run_episode(e_qlearn); q_r.append(r);  q_cf.append(cf)
    e_sarsa.close(); e_qlearn.close()
    
    sm = np.mean(s_r[-50:]); qm = np.mean(q_r[-50:])
    scf = sum(s_cf); qcf = sum(q_cf)
    results[eps_val] = (sm, qm, scf, qcf)
    print(f'{eps_val:>6.2f}  {sm:>14.1f}  {qm:>12.1f}  {scf:>13d}  {qcf:>10d}')

print()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Effect of Exploration Rate ε on SARSA vs Q-Learning', fontsize=12, fontweight='bold')

eps_x = epsilons
sarsa_r_vals  = [results[e][0] for e in epsilons]
qlearn_r_vals = [results[e][1] for e in epsilons]
sarsa_cf_vals  = [results[e][2] for e in epsilons]
qlearn_cf_vals = [results[e][3] for e in epsilons]

ax1.plot(eps_x, sarsa_r_vals,  'o-', color='steelblue', linewidth=2, ms=8, label='SARSA')
ax1.plot(eps_x, qlearn_r_vals, 'o-', color='coral',     linewidth=2, ms=8, label='Q-Learning')
ax1.axhline(-13, color='steelblue', linestyle=':', alpha=0.5)
ax1.axhline(-12, color='coral',     linestyle=':', alpha=0.5)
ax1.set_title('Final Reward vs ε\n(last 50 episodes)')
ax1.set_xlabel('Epsilon (ε)'); ax1.set_ylabel('Mean Reward')
ax1.legend(fontsize=10); ax1.grid(True, alpha=0.3)
ax1.annotate('Gap widens\nwith higher ε', xy=(0.1, np.mean([sarsa_r_vals[2], qlearn_r_vals[2]])),
             fontsize=9, color='gray', ha='center')

ax2.plot(eps_x, sarsa_cf_vals,  'o-', color='steelblue', linewidth=2, ms=8, label='SARSA')
ax2.plot(eps_x, qlearn_cf_vals, 'o-', color='coral',     linewidth=2, ms=8, label='Q-Learning')
ax2.set_title('Total Cliff Falls vs ε\n(200 episodes total)')
ax2.set_xlabel('Epsilon (ε)'); ax2.set_ylabel('Total Cliff Falls')
ax2.legend(fontsize=10); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Cell 12 — Expected SARSA: The Middle Ground

Expected SARSA replaces the sampled `Q(s', a')` with the **expected value** over the current policy:

```
Q(s,a) ← Q(s,a) + α·[r + γ · Σ_{a'} π(a'|s') · Q(s',a') − Q(s,a)]
```

With ε-greedy policy π(a|s):
- Best action (greedy): weight `(1 − ε + ε/|A|)`
- Other actions: weight `ε/|A|`

This removes the variance from sampling `a'` while still accounting for the full ε-greedy distribution.


In [ ]:
class ExpectedSARSAAgent(SARSAAgent):
    """
    Expected SARSA: uses expected Q-value under the current ε-greedy policy.
    
    Reduces variance vs SARSA (no sampling of a')
    but requires computing the full expectation at each update.
    """
    
    def expected_q(self, state):
        """Compute E_π[Q(state, a)] under the ε-greedy policy."""
        n = self.n_actions
        q_vals       = self.Q[state]
        greedy_a     = np.argmax(q_vals)
        
        # ε-greedy probabilities
        probs = np.ones(n) * (self.epsilon / n)
        probs[greedy_a] += (1.0 - self.epsilon)
        
        return np.dot(probs, q_vals)
    
    def run_episode(self, env):
        state, _ = env.reset()
        total_reward = 0
        trajectory   = [state]
        cliff_falls  = 0
        
        for _ in range(10_000):
            action = self.epsilon_greedy(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            # Expected SARSA update: no a_next sampling
            current_q = self.Q[state, action]
            future_q  = 0.0 if done else self.expected_q(next_state)
            td_error  = reward + self.gamma * future_q - current_q
            self.Q[state, action] += self.alpha * td_error
            self.td_errors.append(abs(td_error))
            
            if reward == -100: cliff_falls += 1
            trajectory.append(next_state)
            total_reward += reward
            state = next_state
            if done: break
        
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
        return total_reward, trajectory, cliff_falls


# Train all three
np.random.seed(SEED)
N_EXP = 300
e1 = gym.make('CliffWalking-v1')
e2 = gym.make('CliffWalking-v1')
e3 = gym.make('CliffWalking-v1')

ag_s  = SARSAAgent(N_STATES, N_ACTIONS, alpha=0.5, gamma=0.99, epsilon=0.1)
ag_q  = QLearningAgent(N_STATES, N_ACTIONS, alpha=0.5, gamma=0.99, epsilon=0.1)
ag_es = ExpectedSARSAAgent(N_STATES, N_ACTIONS, alpha=0.5, gamma=0.99, epsilon=0.1)

r_s, r_q, r_es   = [], [], []
cf_s, cf_q, cf_es = [], [], []

for _ in range(N_EXP):
    sr, _, scf  = ag_s.run_episode(e1);  r_s.append(sr);  cf_s.append(scf)
    qr, _, qcf  = ag_q.run_episode(e2);  r_q.append(qr);  cf_q.append(qcf)
    er, _, ecf  = ag_es.run_episode(e3); r_es.append(er); cf_es.append(ecf)

for e in [e1, e2, e3]: e.close()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Three-Way Comparison: SARSA vs Expected SARSA vs Q-Learning', fontsize=13, fontweight='bold')

ep_x = np.arange(N_EXP)
def rm(d, w=20): return [np.mean(d[max(0,i-w):i+1]) for i in range(len(d))]

axes[0].plot(ep_x, rm(r_s),  color='steelblue', linewidth=2, label=f'SARSA ({np.mean(r_s[-50:]):.0f})')
axes[0].plot(ep_x, rm(r_es), color='green',     linewidth=2, label=f'Exp.SARSA ({np.mean(r_es[-50:]):.0f})')
axes[0].plot(ep_x, rm(r_q),  color='coral',     linewidth=2, label=f'Q-Learning ({np.mean(r_q[-50:]):.0f})')
axes[0].axhline(-13, color='gray', linestyle=':', alpha=0.5)
axes[0].set_title('Rolling Mean Reward (20-ep)'); axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Reward'); axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-200, 0)

axes[1].plot(ep_x, np.cumsum(cf_s),  color='steelblue', linewidth=2, label=f'SARSA (Σ={sum(cf_s)})')
axes[1].plot(ep_x, np.cumsum(cf_es), color='green',     linewidth=2, label=f'Exp.SARSA (Σ={sum(cf_es)})')
axes[1].plot(ep_x, np.cumsum(cf_q),  color='coral',     linewidth=2, label=f'Q-Learning (Σ={sum(cf_q)})')
axes[1].set_title('Cumulative Cliff Falls'); axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Total Falls'); axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

# TD error comparison
s_td  = smooth([abs(e) for e in ag_s.td_errors[-3000:]],  30)
es_td = smooth([abs(e) for e in ag_es.td_errors[-3000:]], 30)
q_td  = smooth([abs(e) for e in ag_q.td_errors[-3000:]],  30)
axes[2].plot(s_td,  color='steelblue', linewidth=1.5, alpha=0.9, label='SARSA')
axes[2].plot(es_td, color='green',     linewidth=1.5, alpha=0.9, label='Exp.SARSA')
axes[2].plot(q_td,  color='coral',     linewidth=1.5, alpha=0.9, label='Q-Learning')
axes[2].set_title('|TD Error| (last 3000 updates)\n(Expected SARSA = lowest variance)')
axes[2].set_xlabel('Update step'); axes[2].legend(fontsize=9); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Summary:')
print(f'  SARSA final reward        : {np.mean(r_s[-50:]):.1f}  | cliff falls: {sum(cf_s)}')
print(f'  Expected SARSA final reward: {np.mean(r_es[-50:]):.1f}  | cliff falls: {sum(cf_es)}')
print(f'  Q-Learning final reward   : {np.mean(r_q[-50:]):.1f}  | cliff falls: {sum(cf_q)}')


## Cell 13 — What Happens When ε → 0? (The Convergence Experiment)

If we slowly decay ε to 0, both SARSA and Q-learning should converge to the **same** optimal policy. This cell demonstrates that the behavioral difference is purely a training-time phenomenon, not a fundamental difference in ultimate capability.


In [ ]:
N_DECAY = 600
np.random.seed(SEED)

# Decaying epsilon: 1.0 → 0.01 over 600 episodes
e_decay = gym.make('CliffWalking-v1')
e_nodecay = gym.make('CliffWalking-v1')

ag_decay   = SARSAAgent(N_STATES, N_ACTIONS, alpha=0.3, gamma=0.99,
                         epsilon=1.0, epsilon_decay=0.99, epsilon_min=0.001)
ag_nodecay = SARSAAgent(N_STATES, N_ACTIONS, alpha=0.3, gamma=0.99,
                         epsilon=0.1, epsilon_decay=1.0, epsilon_min=0.1)

r_decay, r_nodecay = [], []
eps_hist = []

for _ in range(N_DECAY):
    rd, _, _ = ag_decay.run_episode(e_decay)
    rn, _, _ = ag_nodecay.run_episode(e_nodecay)
    r_decay.append(rd)
    r_nodecay.append(rn)
    eps_hist.append(ag_decay.epsilon)

e_decay.close(); e_nodecay.close()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('SARSA with Decaying ε vs Fixed ε\n(Convergence to Optimal with ε → 0)',
             fontsize=12, fontweight='bold')

ep_x2 = np.arange(N_DECAY)
axes[0].plot(ep_x2, rm(r_decay,   20), color='green',     linewidth=2, label='SARSA ε=1.0→0.001 (decaying)')
axes[0].plot(ep_x2, rm(r_nodecay, 20), color='steelblue', linewidth=2, label='SARSA ε=0.1 (fixed)')
axes[0].axhline(-13, color='gray', linestyle=':', alpha=0.6, label='Safe path (−13)')
axes[0].axhline(-12, color='red',  linestyle=':', alpha=0.6, label='Optimal path (−12)')
axes[0].set_title('Reward: Decaying ε Converges to Optimal')
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Reward')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3); axes[0].set_ylim(-150, 0)

axes[1].plot(ep_x2, eps_hist, color='green', linewidth=2)
axes[1].set_title('ε Decay Schedule'); axes[1].set_xlabel('Episode')
axes[1].set_ylabel('ε value'); axes[1].grid(True, alpha=0.3)

# Show final policy of decaying agent
policy_decay = ag_decay.get_policy()
plot_grid('SARSA (ε→0) Final Policy\n(Should converge toward risky path)',
          q_values=ag_decay.Q, policy=policy_decay, ax=axes[2])

plt.tight_layout()
plt.show()

print('Insight: with decaying ε, SARSA eventually converges to the optimal (risky) path')
print('because with ε≈0, the exploration risk is negligible and SARSA's updates')
print('converge to the same Q* as Q-learning.')


## Cell 14 — Final Summary: SARSA vs Q-Learning vs Expected SARSA

| Feature | SARSA | Expected SARSA | Q-Learning |
|---------|-------|----------------|------------|
| Policy type | On-policy | On-policy | Off-policy |
| Update target | Q(s', a') sampled from π | E_π[Q(s', ·)] | max Q(s', ·) |
| Variance | Medium | Low (no sampling) | Low (deterministic max) |
| Cliff path (fixed ε) | Safe (risk-aware) | Safe (risk-aware) | Risky (optimal) |
| Training safety | ✓ Fewer cliff falls | ✓ Fewer cliff falls | ✗ More cliff falls |
| Convergence (ε→0) | Optimal path | Optimal path | Optimal path |
| Computational cost | Cheapest | Medium (full expectation) | Cheapest |
| Use when | Training cost matters, real systems | Low-variance updates needed | Maximizing final policy quality |

### The One-Line Summary

**SARSA**: updates toward what it WILL do (including exploration mistakes) → safe, risk-aware  
**Q-learning**: updates toward what it COULD do (perfect greedy) → optimal, ignores exploration cost  
**Expected SARSA**: updates toward what it IS LIKELY to do (expectation over policy) → best of both
